<img src="https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

Supervised Learning
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

## How to use this notebook

Session 4 ended with a feature table. Today you split it, fit models to it, and find out how good they are. Four kinds of activity are marked with **»**:

* **» Predict** — *before* running, decide what the output will be, then run to check. Only used where you have already met the code.
* **» Investigate** — run a working cell, then answer a question about it, from the output or by changing one thing.
* **» Modify** — change a working cell so it answers the next question.
* **» Make** — write the code yourself. The pattern is always somewhere above. **Hint** blocks fold open if you need them: try first.

Part way through, an **Independent practice** slide hands over to you for a long stretch of self-paced work; a **Regroup** slide marks where we come back together.

The notebook is **one environment**: the imports, the tables and the split exist in every cell after them, so run the cells in order. After a restart, or if a cell complains that `df_train` is not defined, use **Runtime → Run before** to catch up. Every practice cell needs only the setup in section 1, the split in section 3 and the prepared table in section 4.

**MIMIC-IV outputs are not in the published slides or notes** beyond summary figures. The data use agreement forbids sharing the data, so you see the rest only when you run the cells yourself. For the same reason, never paste MIMIC rows into an AI assistant or share them; asking about your code is fine.

> Worked solutions, with explanations, are in the solutions notebook, released after the session.

## Welcome

**Goal:** by the end of today you will have fitted models to your feature table and be able to say how good they are, in numbers a clinician could act on.

Session 4 built the table, and promised that imputation and scaling would learn from the training rows only. Today fits models, tests them on patients they have not seen, and keeps that promise. It starts with a talk on what can happen after a model looks good.

### Before we start
1. Open this notebook in Colab: [Open in Colab](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/notebooks/05_Supervised_Learning.ipynb).
2. **File → Save a copy in Drive**, and work in your copy.
3. Run the **setup** cells in section 1 now, so they are ready when the talk ends.

## Intended Learning Outcomes
By the end of this session you will be able to:

1. **Define** supervised learning, and **state** whether a clinical question needs regression or classification.
2. **Explain** why a test set must be split by patient, and why imputation and scaling learn from the training rows only.
3. **Fit** linear and logistic regression models, and **calculate** sensitivity, positive predictive value and AUC on held-out data.
4. **Compare** models with cross-validation, and **distinguish** a real difference from the variation between one split and another.
5. **Justify** a decision threshold for a stated clinical use, and **critique** a model's validation, using the Epic sepsis model as an example.
6. **Construct** a machine learning pipeline that imputes, scales and fits a model, and **test** it on patients it has never seen.

## Session outline
1. Setup, and a talk: model drift and the Epic sepsis model
2. Regression
3. Testing on patients it has not seen
4. Preprocessing pipelines
5. Classification
6. ROC curves and AUC
7. Cross-validation
8. Independent practice: evaluating models of length of stay and in-hospital death
9. Closing

# 1. Setup
The same course views as Session 4. `chartevents_hr` is not needed today; `first_day_lab` is. The first time, Colab asks you to sign in: use the **Google account linked to PhysioNet**.

**Run this once.**

In [ ]:
%pip install pandas_gbq --quiet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas_gbq
import sklearn

# scikit-learn steps return tables with their column names, not bare arrays
sklearn.set_config(transform_output='pandas')

# The Google Cloud project that pays for your queries. After the course, use your own.
PROJECT = 'bowyerlab-mimic-teaching'

def query(sql, project=PROJECT):
    """Run a SQL query on BigQuery and return the result as a DataFrame."""
    return pandas_gbq.read_gbq(sql, project_id=project)

# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.patients
df_patients = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.patients`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.admissions
df_admissions = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.admissions`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_icu.icustays
df_icustays = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.icustays`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_vitalsign
df_vitals = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_vitalsign`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.diagnoses_icd
df_diagnoses = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.diagnoses_icd`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_lab
df_labs = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_lab`")

print(len(df_patients), len(df_admissions), len(df_icustays), len(df_vitals), len(df_diagnoses), len(df_labs))

`df_stays`: one row per ICU stay, as in Session 4, with the first-day haemoglobin and haematocrit added, and `dead_by_24h` for later.

In [ ]:
df_stays = df_icustays.merge(df_admissions, on=['subject_id', 'hadm_id'])
df_stays = df_stays.merge(df_patients, on='subject_id')
df_stays = df_stays.merge(df_vitals, on=['subject_id', 'stay_id'])
df_stays = df_stays.merge(df_labs[['stay_id', 'hemoglobin_min', 'hematocrit_min']], on='stay_id', how='left')

df_stays['died'] = df_stays['hospital_expire_flag'].astype(int)
age = df_stays['anchor_age'] + df_stays['admittime'].dt.year - df_stays['anchor_year']
df_stays['age'] = age.astype(int)
time_to_death = df_stays['deathtime'] - df_stays['intime']
df_stays['dead_by_24h'] = time_to_death.dt.total_seconds() < 24 * 60 * 60

# These columns arrive as Decimal objects (BigQuery NUMERIC), which pandas cannot average: make them floats
for column in ['temperature_min', 'temperature_max', 'temperature_mean', 'hemoglobin_min', 'hematocrit_min', 'los']:
    df_stays[column] = df_stays[column].astype(float)

print(df_stays.shape)

`df_model`: the first ICU stay of each admission, with admission type and race grouped and one-hot encoded as in Session 4's Part 2.

In [ ]:
# The first ICU stay of each admission. Sorted by stay_id so the split below is the same for everyone
df_model = df_stays.sort_values(['intime', 'stay_id'])
df_model = df_model.drop_duplicates('hadm_id', keep='first')
df_model = df_model.sort_values('stay_id')

grouping = {
    'EW EMER.': 'EMERGENCY', 'DIRECT EMER.': 'EMERGENCY',
    'URGENT': 'URGENT',
    'ELECTIVE': 'ELECTIVE', 'SURGICAL SAME DAY ADMISSION': 'ELECTIVE',
    'OBSERVATION ADMIT': 'OBSERVATION', 'EU OBSERVATION': 'OBSERVATION',
    'DIRECT OBSERVATION': 'OBSERVATION', 'AMBULATORY OBSERVATION': 'OBSERVATION',
}
admission_group = df_model['admission_type'].replace(grouping)
adm_dummies = pd.get_dummies(admission_group, prefix='adm').astype(int)

race_group = df_model['race'].str.replace(r' -.*$', '', regex=True)
race_group = race_group.str.replace(' OR ', '/')
race_group = race_group.replace(['UNKNOWN', 'UNABLE TO OBTAIN', 'PATIENT DECLINED TO ANSWER', 'OTHER'], 'UNKNOWN/OTHER')
race_dummies = pd.get_dummies(race_group, prefix='race').astype(int)

df_model = pd.concat([df_model, adm_dummies, race_dummies], axis=1)
print(df_model.shape)

`df_features`: the same cleaning and columns as the Session 4 solutions, with two differences. It is **not scaled**, and `temperature_mean` **still has its gaps**, because those steps learn from rows and the split has not happened yet.

In [ ]:
# Clinical limits are fixed numbers, not learnt from the rows, so they can come before the split
df_model['glucose_max'] = df_model['glucose_max'].mask(df_model['glucose_max'] > 2000)
df_model['heart_rate_max'] = df_model['heart_rate_max'].mask(df_model['heart_rate_max'] > 250)

temperature_gap = df_model['temperature_mean'].isna()
df_model['temperature_missing'] = temperature_gap.astype(int)

is_icd9_diabetes = df_diagnoses['icd_code'].str.startswith('250')
is_icd10_diabetes = df_diagnoses['icd_code'].str.startswith('E10') | df_diagnoses['icd_code'].str.startswith('E11')
df_diabetes = df_diagnoses[is_icd9_diabetes | is_icd10_diabetes]
has_code = df_model['hadm_id'].isin(df_diabetes['hadm_id'])
df_model['diabetes'] = has_code.astype(int)

numeric_features = ['age', 'heart_rate_mean', 'heart_rate_max', 'resp_rate_mean', 'sbp_min', 'glucose_max']
binary_features = ['temperature_missing', 'diabetes'] + list(adm_dummies.columns) + list(race_dummies.columns)
other_columns = ['subject_id', 'anchor_year_group', 'died', 'los', 'dead_by_24h', 'hemoglobin_min', 'hematocrit_min', 'temperature_mean']

# The model's columns: Session 4's, without diabetes. Section 4 asks why
kept_binary = binary_features.copy()
kept_binary.remove('diabetes')
features = numeric_features + ['temperature_mean'] + kept_binary

df_features = df_model[other_columns + numeric_features + binary_features].copy()
df_features = df_features.dropna(subset=numeric_features)
print(df_features.shape)
print('Stays that ended in death:', round(df_features['died'].mean(), 3))

## Talk: model drift and the Epic sepsis model
Five minutes from this week's trio.

**While they talk, write down one question you would ask them.** Afterwards, two or three of you will ask yours.

This will touch on some things we haven't covered yet, you do not need to worry about details yet.

## Supervised learning
A model learns from rows whose **label** is already known, here whether each patient died, then predicts the label for patients where it is not yet known. **Unsupervised** learning, in Session 7, finds structure in rows that have no label.

The stages of supervised learning, and where this course covers them:

| Acquisition | Preparation | **Training** | **Evaluation** | Deployment |
| :-: | :-: | :-: | :-: | :-: |
| Session 3 | Session 4 | **Today** | **Today** | The talk |

# 2. Regression
Supervised learning has two main forms:

*   **Regression** predicts a number, such as length of stay
*   **Classification** predicts a category, such as died or survived

**Fitting a line** is the simplest regression: $y = mx + c$, where the training data choose the slope $m$ and intercept $c$.

![Regression vs Classification](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-04_Regression-vs-Classification.svg)

## Which line?
*   Through **two points** there is exactly one line: two equations, two unknowns, solve for $m$ and $c$
*   Through **forty points** there is none: real data are noisy, so no line passes through them all. The exact line through any two of them misses the other 38, and which two you pick changes the line
*   So fitting becomes a **minimisation problem**: choose the $m$ and $c$ that make the errors, the red segments, as small as possible overall. **Ordinary least squares** minimises the **sum of squared errors**, $\sum_i (y_i - \hat{y}_i)^2$. The maths is in the appendix.

![Two points, forty points, and least squares](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-04_Which-Line.svg)

## scikit-learn
The Python library for machine learning in this course. Every model in it, from a straight line to a neural network, is used the same way:

```python
model = LinearRegression()        # 1. create it, choosing its settings
model.fit(X, y)                   # 2. fit: learn from the training rows
predictions = model.predict(X)    # 3. predict: apply it to rows
```

*   `X` is a **table** of features, one row per patient, even if it has only one column
*   `y` is one **column**: the label for each of those rows
*   After `fit`, what the model learned is stored on it: for a line, `coef_` (the slope) and `intercept_`

## A worked example: haemoglobin from haematocrit
Some point-of-care analysers used in theatre measure **haematocrit** and calculate **haemoglobin** from it, and transfusion decisions are made on haemoglobin. The general rule of thumb, the **rule of three**, says haemoglobin is about a third of haematocrit. Can a model find it?

The data: `hematocrit_min` and `hemoglobin_min` in `df_features`, each patient's lowest value in their first ICU day, from the `first_day_lab` table. Rows missing either value are dropped, so there are no gaps to fill. `[['hematocrit_min']]`, with double brackets, keeps the column a table: that is `X`.

In [ ]:
from sklearn.linear_model import LinearRegression

blood_rows = df_features.dropna(subset=['hemoglobin_min', 'hematocrit_min'])

model_blood = LinearRegression()
model_blood.fit(blood_rows[['hematocrit_min']], blood_rows['hemoglobin_min'])

print('Slope:    ', round(model_blood.coef_[0], 3))
print('Intercept:', round(model_blood.intercept_, 3))

**» Investigate** — **Does the slope agree with the rule of three?**

## How good is it?
`predict` applies the fitted line. The **root mean squared error** (RMSE) is the typical size of the error, in the units of the output. A number on its own means little, so we compare it with the simplest possible model: **predict the mean for everyone**.

In [ ]:
from sklearn.metrics import root_mean_squared_error

predicted_rows = model_blood.predict(blood_rows[['hematocrit_min']])
# The baseline: the same prediction, the mean, for every patient
mean_haemoglobin = blood_rows['hemoglobin_min'].mean()
mean_rows = np.full(len(blood_rows), mean_haemoglobin)

print('RMSE of the model:           ', round(root_mean_squared_error(blood_rows['hemoglobin_min'], predicted_rows), 2), 'g/dL')
print('RMSE of predicting the mean: ', round(root_mean_squared_error(blood_rows['hemoglobin_min'], mean_rows), 2), 'g/dL')

## What the RMSE measures
The RMSE is the typical length of a red segment. The baseline predicts the mean for everyone: a flat line that ignores haematocrit. A model is only worth having if its segments are shorter.

Each point is the average of 10 patients with similar haematocrit. The data use agreement does not let us publish one patient's values, and these slides are public. Averaging also shortens the segments, so they are smaller than one patient's error; the RMSEs in the titles are for every patient.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error

# This figure is published, and the MIMIC data use agreement forbids sharing individual patients' values.
# So each point is the average of 10 patients with similar haematocrit, the usual small-numbers threshold.
sorted_rows = blood_rows.sort_values('hematocrit_min')
group = np.arange(len(sorted_rows)) // 10
group_means = sorted_rows.groupby(group)[['hematocrit_min', 'hemoglobin_min']].mean()
group_sizes = sorted_rows.groupby(group).size()
group_means = group_means[group_sizes == 10]

# Sixty of the groups, so the segments can be seen. random_state fixes which sixty, so the figure is the same every run
shown = group_means.sample(60, random_state=0)
shown = shown.sort_values('hematocrit_min')

flat = np.full(len(shown), blood_rows['hemoglobin_min'].mean())
fitted = model_blood.predict(shown[['hematocrit_min']])
panels = [
    ('Predict the mean for everyone', flat, root_mean_squared_error(blood_rows['hemoglobin_min'], mean_rows)),
    ('The fitted line', fitted, root_mean_squared_error(blood_rows['hemoglobin_min'], predicted_rows)),
]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, (name, predicted, rmse) in zip(axes, panels):
    ax.vlines(shown['hematocrit_min'], predicted, shown['hemoglobin_min'], color='#c0392b', linewidth=1)
    ax.scatter(shown['hematocrit_min'], shown['hemoglobin_min'], color='#444444', s=22, zorder=3)
    ax.plot(shown['hematocrit_min'], predicted, color='#0000cd', linewidth=2)
    ax.set_title(f'{name}: RMSE {rmse:.2f} g/dL', fontsize=10)
    ax.set_xlabel('Lowest first-day haematocrit (%)')
    for side in ['top', 'right']:
        ax.spines[side].set_visible(False)
axes[0].set_ylabel('Lowest first-day haemoglobin (g/dL)')
plt.tight_layout()
plt.show()

**» Investigate** — The model learned from `blood_rows`, and the RMSE is computed on `blood_rows` too. In Session 4, Model C had almost no error on the rows it learned from and did badly on new patients. **Would you tell a clinician this model will be about 0.59 g/dL out for their next patient? What would you need to check first?**

# 3. Testing on patients it has not seen
*   The answer to that question: check the model on patients it has **never seen**. Model C in Session 4 shows why: almost no error on the rows it learned from, and badly wrong on new patients
*   We do not have next year's patients, so we hold some of today's back: a **training set** to fit on, and a **test set** that stands in for the next patients, typically 80/20 or 70/30
*   From here on, everything that learns from rows learns from the training set alone

![Illustration of data modelling](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-04_Train-Test-Split.svg)

## Splitting the rows
`train_test_split` shuffles a table and cuts it in two. `random_state` fixes the shuffle, so everyone gets the same split.

In [ ]:
from sklearn.model_selection import train_test_split

rows_train, rows_test = train_test_split(df_features, test_size=0.2, random_state=123)

print(len(rows_train), 'training rows,', len(rows_test), 'test rows')

**» Investigate** — Each row is one admission, and some patients were admitted more than once. **What might happen when we split these rows? Why would that matter for a model meant for new patients?**

## Splitting the patients
First, the check: how many test admissions belong to a patient who is also in the training rows?

In [ ]:
also_in_train = rows_test['subject_id'].isin(rows_train['subject_id'])
print(also_in_train.sum(), 'of', len(rows_test), 'test rows belong to a patient who is also in the training rows')

The fix: split the **patients**, then take each patient's rows with `isin`, as you did for diabetes in Session 4.

In [ ]:
# 1. The list of patients, each one once
patients = df_features['subject_id'].unique()

# 2. Split the patients, NOT THE ROWS: 80% to training, 20% to test
train_patients, test_patients = train_test_split(patients, test_size=0.2, random_state=123)

# 3. Each patient's rows follow them: True for rows whose patient is in that list
in_train = df_features['subject_id'].isin(train_patients)
in_test = df_features['subject_id'].isin(test_patients)

df_train = df_features[in_train].copy()
df_test = df_features[in_test].copy()

print(len(df_train), 'training rows,', len(df_test), 'test rows')

Two checks: no test patient is in the training rows now, and the death rate is similar on both sides.

In [ ]:
also_in_train = df_test['subject_id'].isin(df_train['subject_id'])
print(also_in_train.sum(), 'test rows belong to a patient in the training rows')

print('Died, training rows:', round(df_train['died'].mean(), 3))
print('Died, test rows:    ', round(df_test['died'].mean(), 3))

## Fit on the training patients
The same model, fitted again on the training patients only.

In [ ]:
blood_train = df_train.dropna(subset=['hemoglobin_min', 'hematocrit_min'])
blood_test = df_test.dropna(subset=['hemoglobin_min', 'hematocrit_min'])

model_blood = LinearRegression()
model_blood.fit(blood_train[['hematocrit_min']], blood_train['hemoglobin_min'])

print('Slope:    ', round(model_blood.coef_[0], 3))
print('Intercept:', round(model_blood.intercept_, 3))

## How good is it, on patients it has not seen?
The same RMSE and baseline, now on the test patients. The baseline is the **training** mean, because that is all we would know when the next patient arrives.

In [ ]:
from sklearn.metrics import root_mean_squared_error

predicted_blood = model_blood.predict(blood_test[['hematocrit_min']])
# The baseline: the training mean, for every test patient
mean_haemoglobin = blood_train['hemoglobin_min'].mean()
mean_blood = np.full(len(blood_test), mean_haemoglobin)

print('RMSE of the model:           ', round(root_mean_squared_error(blood_test['hemoglobin_min'], predicted_blood), 2), 'g/dL')
print('RMSE of predicting the mean: ', round(root_mean_squared_error(blood_test['hemoglobin_min'], mean_blood), 2), 'g/dL')

**» Investigate** — **Compare these RMSEs with the ones in section 2. Why is the gap so much smaller than Model C's in Session 4?**

The fitted line through the test patients, when you run the next cell. Plots of individual patients stay in your notebook.

In [ ]:
plt.scatter(blood_test['hematocrit_min'], blood_test['hemoglobin_min'], s=4, alpha=0.4, label='Test patients')
plt.plot(blood_test['hematocrit_min'], predicted_blood, color='#c0392b', label='The fitted line')
plt.xlabel('Lowest first-day haematocrit (%)')
plt.ylabel('Lowest first-day haemoglobin (g/dL)')
plt.legend()
plt.show()

This one is easy: the two tests measure nearly the same thing. In the practice you will use exactly the same steps on something that matters, length of stay, and find out how much harder it is.

# 4. Preprocessing pipelines
The preparation steps, learned from the training rows and applied to both.

![Preprocessing pipeline: fitted on the training rows, applied to both](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-04_Preprocessing-Pipeline.svg)

## Preparation comes after the split
*   Two steps from Session 4 **learn from rows**: imputation learns a median, and scaling learns a mean and a spread
*   So both come **after** the split, and learn from the **training** rows only
*   The test rows are then changed with what was learned, never learned from: a deployed model cannot learn from patients who have not arrived yet

## scikit-learn's pipelines
`make_pipeline` chains steps into one object:

*   `fit` learns every step from the rows it is given
*   `transform` applies what it learned, to any rows

A pipeline can end with a model too, so one `fit` learns the preparation and the model together. Section 7 does that.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

prepare = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
prepare.fit(df_train[features])

X_train = prepare.transform(df_train[features])
X_test = prepare.transform(df_test[features])

**» Investigate** — Once deployed, the model sees **one patient at a time**. **Why can't it scale that patient with their own mean and spread? Where do the numbers it uses come from?**

`diabetes` comes from `diagnoses_icd`, which is coded after discharge. Session 4 section 5 flagged it, so it does not go into the model.

# 5. Classification
**Logistic regression** starts with the same weighted sum of features as linear regression, then squashes it through the **sigmoid** so the output lies between 0 and 1: a probability. A **threshold**, 0.5 by default, turns the probability into a yes or no.

![The sigmoid, with the 0.5 threshold](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-04_Sigmoid.svg)

## Fitting it
The same two steps as the regression, on the prepared table from section 4.

A straight line has a formula for its best slope and intercept. With the sigmoid in the way there is no formula, so `LogisticRegression` searches: it starts with a guess and improves it until it stops changing. The appendix has the details.

In [ ]:
from sklearn.linear_model import LogisticRegression

y_train = df_train['died']
y_test = df_test['died']

model_died = LogisticRegression()
model_died.fit(X_train, y_train)

# Column 1 is the probability of the label 1, died
prob_test = model_died.predict_proba(X_test)[:, 1]
pred_test = model_died.predict(X_test)

print('Predicted probability, median:', round(np.median(prob_test), 3))
print('Patients predicted to die:', pred_test.sum(), 'of', len(pred_test))

**» Predict** — In Session 4, a model that says "survives" for everyone was right for about 88% of stays. This model has age, vital signs and admission type. **Write down the accuracy you expect it to reach.**

In [ ]:
from sklearn.metrics import accuracy_score

print('Always "survives":', round(1 - y_test.mean(), 3))
print('The model:        ', round(accuracy_score(y_test, pred_test), 3))

## The confusion matrix
Accuracy adds two kinds of correct answer and hides two kinds of mistake. The **confusion matrix** counts them separately.

![Illustration of a confusion matrix](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-04_Confusion-Matrix.svg)

The confusion matrix for our model, on the test patients:

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

matrix = confusion_matrix(y_test, pred_test)
display = ConfusionMatrixDisplay(matrix, display_labels=['Survived', 'Died'])
display.plot(colorbar=False)
plt.show()

**» Investigate** — Each row of the matrix is what actually happened; each column is what the model predicted. The bottom row is the patients who died. **How many of them did the model flag, and how many did it miss?** Would you switch this on?

## Measures from the matrix

| Measure | Formula | The question it answers |
| :-- | :-- | :-- |
| **Accuracy** | (TP + TN) / everyone | How many are right? |
| **Sensitivity** (recall) | TP / (TP + FN) | Of those who died, how many did it flag? |
| **Positive predictive value** (PPV, precision) | TP / (TP + FP) | Of those flagged, how many died? |
| **F1 score** | 2 × PPV × sensitivity / (PPV + sensitivity) | One number balancing the two |

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print('Accuracy:   ', round(accuracy_score(y_test, pred_test), 2))
print('Sensitivity:', round(recall_score(y_test, pred_test), 2))
print('PPV:        ', round(precision_score(y_test, pred_test), 2))
print('F1:         ', round(f1_score(y_test, pred_test), 2))

## Moving the threshold
Back to the sigmoid. `predict` turned each probability into died or survived at **0.5**, a threshold we never chose: the 32 flagged and the 24 caught both came from it.

![The sigmoid, with the 0.5 threshold](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-04_Sigmoid.svg)

We can move it. This cell flags patients from the probabilities, `prob_test`, at whatever threshold we set. Where to put it is a clinical decision: what a missed patient costs against what a false alarm costs.

In [ ]:
threshold = 0.5
flagged = prob_test >= threshold

print('Sensitivity:     ', round(recall_score(y_test, flagged), 2))
print('PPV:             ', round(precision_score(y_test, flagged), 2))
print('Accuracy:        ', round(accuracy_score(y_test, flagged), 2))
print('Flagged per 100: ', round(100 * flagged.mean()))

**» Investigate** — **Change `threshold` to 0.1 and run it again.** What happens to each of the four numbers? Which one makes the lower threshold look worst, and is that the one a clinician should care about?

# 6. ROC curves and AUC
Every threshold gives a different confusion matrix. We would like to judge the model **before choosing one**. Ten made-up patients, three of whom died, with the score a model gave each.

Two rates describe each threshold: the **true positive rate** (TPR), which is the sensitivity, and the **false positive rate** (FPR), the share of survivors flagged.

![Ten made-up patients at three thresholds](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-04_ROC-Thresholds.svg)

**» Investigate** — Two rows are done. **Work out the third from the right-hand panel.**

| Threshold | Died and flagged | TPR | Survived and flagged | FPR |
| --: | --: | --: | --: | --: |
| 0.7 | 1 of 3 | 0.33 | 1 of 7 | 0.14 |
| 0.4 | 2 of 3 | 0.67 | 2 of 7 | 0.29 |
| 0.15 | ? | ? | ? | ? |

## The ROC curve
Plot FPR against TPR at **every** threshold. The third point is the one you just worked out: 3 of 3 and 5 of 7.

![The ROC curve for the ten patients](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-04_ROC-Curve.svg)

## The area under it
*   A **high** threshold flags nobody: the bottom left. A **low** threshold flags everybody: the top right
*   The dashed diagonal is a model that scores at random: AUC 0.5. A perfect model reaches the top left corner: AUC 1
*   **AUC has a plain meaning: pick one patient who died and one who survived. It is the chance the model scored the one who died higher**

Here there are 3 × 7 = 21 such pairs. The model gets 18 right, so the AUC is 18 / 21 = 0.86.

## On the real model
The same curve for `model_died`, with the AUC on the test rows and on the training rows.

In [ ]:
from sklearn.metrics import roc_auc_score, RocCurveDisplay

print('AUC, test rows:    ', round(roc_auc_score(y_test, prob_test), 3))
prob_train = model_died.predict_proba(X_train)[:, 1]
print('AUC, training rows:', round(roc_auc_score(y_train, prob_train), 3))

fig, ax = plt.subplots(figsize=(4, 4))
RocCurveDisplay.from_predictions(y_test, prob_test, plot_chance_level=True, ax=ax)
plt.show()

**» Investigate** — Accuracy barely beat "always survives", yet the AUC is about 0.8. **How can both be true?** Think about what each one depends on.

## Ranking versus deciding
*   Most patients who died got a raised probability, but only 24 of 197 reached 0.5
*   **Accuracy** judges the decisions at 0.5 only, so it barely moves
*   **AUC** judges the order: a patient who died outranks one who survived 80% of the time
*   Moving the threshold uses that ranking (Part 3). PPV then says whether the alerts are worth acting on

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Published, so each bar counts patients: bins of 0.1, everything from 0.5 up in one bar, and no bar under 10 patients
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0]
labels = ['0–0.1', '0.1–0.2', '0.2–0.3', '0.3–0.4', '0.4–0.5', '0.5 and up']
groups = [('Survived', prob_test[y_test == 0], '#999999'), ('Died', prob_test[y_test == 1], '#c0392b')]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, (name, probs, colour) in zip(axes, groups):
    counts, _ = np.histogram(probs, bins=bins)
    shown = np.where(counts >= 10, counts, 0)
    ax.bar(range(len(labels)), 100 * shown / len(probs), color=colour, edgecolor='white')
    ax.axvline(4.5, color='#0000cd', linestyle='--', linewidth=1.5)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_title(f'{name}: {len(probs)} test patients', fontsize=10)
    ax.set_xlabel('Predicted probability of death')
    for side in ['top', 'right']:
        ax.spines[side].set_visible(False)
axes[0].set_ylabel('% of patients')
axes[0].annotate('threshold 0.5', xy=(4.6, 60), color='#0000cd', fontsize=9)
plt.tight_layout()
plt.show()

## Calibration: is 20% really 20%?
Three different questions about a risk model:

1. **Ranking:** are sicker patients scored higher? The AUC
2. **Deciding:** at what risk do we act? The threshold, with sensitivity and PPV
3. **Calibration:** when the model says 20%, do about 20% die? A clinician telling a family "about one in five" needs this one

Each point is a tenth of the test patients, about 170. Here the model sits close to the line. Calibration often slips while the AUC holds, when a model moves to another hospital or another year: the Closing comes back to it.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

# Ten bands of equal size, about 170 test patients each: the mean predicted risk against the share who died
observed, predicted = calibration_curve(y_test, prob_test, n_bins=10, strategy='quantile')

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot([0, 0.6], [0, 0.6], color='#999999', linestyle='--', label='Perfect calibration')
ax.plot(predicted, observed, marker='o', color='#c0392b', label='The model, on the test patients')
ax.set_xlabel('Mean predicted risk of death, in each band')
ax.set_ylabel('Share who died, in each band')
ax.set_xlim(0, 0.6)
ax.set_ylim(0, 0.6)
ax.legend(frameon=False, fontsize=9)
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

# 7. Cross-validation
The AUC in section 6 came from one split of the patients. Here is the same model on thirty other splits. Each split needs its own preparation, so the model goes on the end of section 4's preprocessing pipeline: one `fit` learns the imputer, the scaler and the model from that split's training rows.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(),
)

split_aucs = []
for seed in range(30):
    patients_a, patients_b = train_test_split(patients, test_size=0.2, random_state=seed)
    rows_a = df_features[df_features['subject_id'].isin(patients_a)]
    rows_b = df_features[df_features['subject_id'].isin(patients_b)]

    pipeline.fit(rows_a[features], rows_a['died'])
    prob_b = pipeline.predict_proba(rows_b[features])[:, 1]
    split_aucs.append(roc_auc_score(rows_b['died'], prob_b))

print('The same model, 30 splits: AUC from', round(min(split_aucs), 3), 'to', round(max(split_aucs), 3))

Each dot is the same model on one split. The arrow is the gain a paper might claim.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# One dot per split; the small vertical scatter only keeps dots from hiding each other
rng = np.random.default_rng(0)
height = rng.uniform(-0.3, 0.3, len(split_aucs))

fig, ax = plt.subplots(figsize=(9, 1.8))
ax.scatter(split_aucs, height, color='#0000cd', s=30, alpha=0.7)
ax.annotate('', xy=(0.80, 0.55), xytext=(0.78, 0.55), arrowprops=dict(arrowstyle='<->', color='#c0392b'))
ax.text(0.79, 0.65, 'a gain of 0.02', ha='center', color='#c0392b', fontsize=9)
ax.set_ylim(-0.6, 1.0)
ax.set_yticks([])
ax.set_xlabel('AUC on the test patients, one dot per split')
for side in ['top', 'right', 'left']:
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

**» Investigate** — A paper reports that its new model beats the old one: AUC **0.80 against 0.78**, on one test set. **Looking at these dots, what would you want to see before believing it?**

## k-fold cross-validation
Split the **training** patients into $k$ folds. Train on all but one, evaluate on the one left out, and repeat so each fold has a turn. The test set stays out of it.

![Five-fold cross-validation](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-04_K-Fold.svg)

`GroupKFold` keeps each patient in one fold, as the split in section 3 did. Every fold gets its own imputer and scaler, because the pipeline is refitted each time.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_score

scores = cross_val_score(
    pipeline,
    df_train[features],
    df_train['died'],
    cv=GroupKFold(n_splits=5),
    groups=df_train['subject_id'],
    scoring='roc_auc',
)
print('AUC per fold:', scores.round(3))
print('Mean AUC:    ', round(scores.mean(), 3))

## Other classifiers
Every scikit-learn classifier has the same `fit` and `predict`, so you can swap one into the pipeline in the practice. The appendix has a slide on each.

| Model | The idea | scikit-learn |
| :-- | :-- | :-- |
| **Decision tree** | A series of yes/no splits on features | `DecisionTreeClassifier` |
| **K-nearest neighbours** | What happened to the most similar patients? | `KNeighborsClassifier` |
| **Support vector machine** | A boundary as far as possible from both groups | `SVC` |

**Compare models with cross-validation on the training rows; use the test set once, at the end.** Choosing a model by its test score makes the test score optimistic.

Settings you choose before fitting, such as a tree's depth or the number of neighbours, are **hyperparameters**. Choosing them is what Part 6 does by hand: compare them by cross-validation on the training rows, never by the test set. Session 6 automates it with `GridSearchCV`.

# 8. Independent practice: evaluating models
From here to the **Regroup** is yours, in your pair: about 75 minutes.

* Each task is marked **» Modify** or **» Make**, and most ask for a **comment** recording a decision. Those comments are what we revisit in the Closing.
* **Hint** blocks fold open: try first.
* Every cell needs only the setup in section 1, the split in section 3 and the prepared table in section 4.

Work together, and call me over when you get stuck. **Understanding beats finishing.**

## Part 1: Regression on something that matters
**» Modify** — This cell predicts ICU length of stay (`los`, in days) from age, and compares it with predicting the mean. **Change it to use all of `features`.** In the comment: how much better than the mean is it, in days? Don't expect haemoglobin levels of performance: if your model barely beats the mean, that may be the right answer, and the comment is where you say why.

<details><summary><b>Hint</b></summary>

`X_train[features]` is the same as `X_train`. Change the two places the model is given its columns, in `fit` and in `predict`.

</details>

In [ ]:
# How much better than predicting the mean, in days:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

model_los = LinearRegression()
model_los.fit(X_train[['age']], df_train['los'])
predicted_los = model_los.predict(X_test[['age']])

# The baseline: the training mean, for every test patient
mean_stay = df_train['los'].mean()
mean_los = np.full(len(df_test), mean_stay)
print('Model:            ', round(root_mean_squared_error(df_test['los'], predicted_los), 2), 'days')
print('Predicting the mean:', round(root_mean_squared_error(df_test['los'], mean_los), 2), 'days')

**» Make** — **R-squared** is a second regression metric, alongside the RMSE. The RMSE is in the units of the output; R-squared turns the comparison with predicting the mean into one number without units, the share of the variation the model explains: 1 is perfect, 0 is no better than the mean, below 0 is worse. **Compute it for your model of `los`, with `r2_score` from `sklearn.metrics`, on the test rows.** Then plot the actual `los` against the predicted `los`. In the comment: why might length of stay be so much harder than haemoglobin?

<details><summary><b>Hint</b></summary>

`r2_score(actual, predicted)` takes the same two arguments as `root_mean_squared_error`. `plt.scatter(predicted_los, df_test['los'], s=4)`, then label the axes. Think about what happens to a patient after the first 24 hours, and whether anything in the table knows it.

</details>

In [ ]:
# Why harder than haemoglobin:
from sklearn.metrics import r2_score

r2 = r2_score(____, ____)
print('R-squared:', round(r2, 3))

Then the plot: actual length of stay against predicted. It stays in your notebook.

In [ ]:
plt.scatter(____, ____, s=4)
plt.xlabel('Predicted length of stay (days)')
plt.ylabel('Actual length of stay (days)')
plt.show()

## Part 2: A full evaluation
**» Modify** — This cell fits a classifier on age alone and gives its AUC. **Add the two vital signs you chose in Session 4**, one at a time. How much does each add?

<details><summary><b>Hint</b></summary>

Add column names to the `chosen` list, such as `'sbp_min'` or `'resp_rate_mean'`. Nothing else needs to change.

</details>

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

chosen = ['age']

model_chosen = LogisticRegression()
model_chosen.fit(X_train[chosen], df_train['died'])
prob_chosen = model_chosen.predict_proba(X_test[chosen])[:, 1]

print(chosen, 'AUC', round(roc_auc_score(df_test['died'], prob_chosen), 3))

**» Make** — **A full evaluation of a model on all of `features`:** a confusion matrix at the default threshold, sensitivity, PPV and AUC, on the test rows. In the comment: **is it better than "always survives", and by which measures?**

<details><summary><b>Hint</b></summary>

Section 5 has every line: `LogisticRegression()`, `fit`, `predict`, `predict_proba(...)[:, 1]`, then `confusion_matrix`, `recall_score`, `precision_score`, `roc_auc_score`. Give your model its own name, so that you can re-run the cell on its own.

</details>

In [ ]:
# Better than "always survives"? By which measures:

# 1. Fit a logistic regression on X_train, and predict on X_test

# 2. The confusion matrix

# 3. Sensitivity, PPV and AUC

## Part 3: Choosing a threshold
**» Modify** — A rapid-response team wants to catch **at least 80% of the patients who will die**. **Change `threshold` until the sensitivity reaches 0.8.** In the comment: how many patients per 100 does the team review, and how many of those flags are true? Would you deploy this?

In [ ]:
# Reviews per 100 patients, how many are true, and would you deploy it:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

model_alert = LogisticRegression()
model_alert.fit(X_train, df_train['died'])
prob_alert = model_alert.predict_proba(X_test)[:, 1]

threshold = 0.5
flagged = prob_alert >= threshold

print('Sensitivity:    ', round(recall_score(df_test['died'], flagged), 2))
print('PPV:            ', round(precision_score(df_test['died'], flagged), 2))
print('Flagged per 100:', round(100 * flagged.mean()))

## Part 4: Still ventilated after surgery?
A patient still on a ventilator two days after surgery still needs their ICU bed, and tomorrow's operating list may depend on that bed. **Can we tell at 24 hours who will still be ventilated at 48?**

Two new tables: `services`, the team caring for the patient, and `ventilation`, each period of ventilation in the ICU. Run the next three cells to build the question.

In [ ]:
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.services
df_services = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.services`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.ventilation
df_ventilation = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.ventilation`")

print(df_services['curr_service'].value_counts().head(8))

**Who counts as surgical:** any patient cared for by a surgical team (cardiac, general, vascular, neuro, trauma or orthopaedic surgery) by the time we predict, 24 hours after ICU admission. Team changes after that are ignored: at 24 hours we would not know about them yet.

In [ ]:
surgical_services = ['CSURG', 'SURG', 'VSURG', 'NSURG', 'TSURG', 'ORTHO']

services = df_services.merge(df_model[['hadm_id', 'intime']], on='hadm_id')
prediction_time = services['intime'] + pd.Timedelta(hours=24)
known_by_then = services['transfertime'] <= prediction_time
is_surgical = services['curr_service'].isin(surgical_services)

surgical = services[known_by_then & is_surgical]
df_model['surgical'] = df_model['hadm_id'].isin(surgical['hadm_id'])
print(df_model['surgical'].sum(), 'surgical stays')

**The label and one new feature:** on invasive ventilation 48 hours after ICU admission (`vent_48h`), and at 24 hours (`vent_24h`), which is known when the prediction is made.

In [ ]:
invasive_status = ['InvasiveVent', 'Tracheostomy']
invasive = df_ventilation[df_ventilation['ventilation_status'].isin(invasive_status)]
invasive = invasive.merge(df_model[['stay_id', 'intime']], on='stay_id')

for hours in [24, 48]:
    moment = invasive['intime'] + pd.Timedelta(hours=hours)
    covers_moment = (invasive['starttime'] <= moment) & (invasive['endtime'] >= moment)
    ventilated = invasive[covers_moment]
    df_model[f'vent_{hours}h'] = df_model['stay_id'].isin(ventilated['stay_id']).astype(int)

# Surgical patients still in the ICU at 24 hours, kept on the same side of the split as in section 3
in_scope = df_model['surgical'] & (df_model['los'] >= 1)
surgery_train = df_train[in_scope.loc[df_train.index]].copy()
surgery_test = df_test[in_scope.loc[df_test.index]].copy()
for table in [surgery_train, surgery_test]:
    table['vent_24h'] = df_model.loc[table.index, 'vent_24h']
    table['vent_48h'] = df_model.loc[table.index, 'vent_48h']

print(len(surgery_train), 'training rows,', len(surgery_test), 'test rows')
print('Still ventilated at 48 hours:', round(surgery_train['vent_48h'].mean(), 3))

**» Make** — **A full evaluation for `vent_48h`.** Fit a pipeline on `surgery_train[features]`, and report the AUC on `surgery_test`, with the sensitivity and PPV at a threshold you choose. Then **add `vent_24h` to the features** and compare. In the comment: how much does `vent_24h` add, and is it fair to use? Would the team planning tomorrow's list find the model useful?

<details><summary><b>Hint</b></summary>

The pipeline in section 7 has every step: `make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression())`. The features with `vent_24h` added are `features + ['vent_24h']`. Ask whether a column is known at 24 hours, as Session 4 asked of every feature.

</details>

In [ ]:
# How much vent_24h adds, is it fair, and would the team find it useful:

# 1. A pipeline: imputer, scaler, logistic regression. Fit it on surgery_train[features] and surgery_train['vent_48h']

# 2. Predicted probabilities for surgery_test, and the AUC

# 3. Sensitivity and PPV at a threshold you choose

# 4. The same again, with features + ['vent_24h']

## Part 5: Patients who were already dead
Our prediction is made 24 hours after ICU admission, but `df_features` includes patients who had died by then: `dead_by_24h` is `True` for them. Session 4's stretch task raised this.

**» Modify** — This cell gives the AUC on every row. **Change the two `rows_` lines to keep only patients with `dead_by_24h` equal to `False`.** In the comment: what happens to the AUC, and which number is the honest one for a model used at 24 hours?

<details><summary><b>Hint</b></summary>

`df_train[df_train['dead_by_24h'] == False]`, and the same for `df_test`.

</details>

In [ ]:
# What happens to the AUC, and which number is the honest one:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rows_train_kept = df_train
rows_test_kept = df_test

pipeline_rows = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression())
pipeline_rows.fit(rows_train_kept[features], rows_train_kept['died'])
prob_rows = pipeline_rows.predict_proba(rows_test_kept[features])[:, 1]

print(len(rows_test_kept), 'test rows, AUC', round(roc_auc_score(rows_test_kept['died'], prob_rows), 3))

## Part 6: Comparing models
**» Modify** — This cell cross-validates logistic regression on the training rows. **Change `candidate`, one at a time, to `DecisionTreeClassifier(max_depth=5)`, `KNeighborsClassifier(n_neighbors=50)` and `SVC()`.** In the comment: which would you take forward, and is the gap bigger than the spread between folds?

In [ ]:
# Which model, and is the gap bigger than the spread between folds:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

candidate = LogisticRegression()

pipeline_candidate = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), candidate)
scores = cross_val_score(
    pipeline_candidate,
    df_train[features],
    df_train['died'],
    cv=GroupKFold(n_splits=5),
    groups=df_train['subject_id'],
    scoring='roc_auc',
)
print('Mean AUC', round(scores.mean(), 3), ' spread between folds', round(scores.std(), 3))

**» Modify** — A tree of depth 5, on the training rows and the test rows. **Change `max_depth` to `None`**, which lets the tree grow until it fits every training row. Which of Session 4's Models A, B and C is this, and how can you tell from the two numbers?

In [ ]:
# Which of Session 4's models, and how you can tell:
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

tree = make_pipeline(SimpleImputer(strategy='median'), DecisionTreeClassifier(max_depth=5, random_state=0))
tree.fit(df_train[features], df_train['died'])

prob_tree_train = tree.predict_proba(df_train[features])[:, 1]
prob_tree_test = tree.predict_proba(df_test[features])[:, 1]
print('AUC, training rows:', round(roc_auc_score(df_train['died'], prob_tree_train), 3))
print('AUC, test rows:    ', round(roc_auc_score(df_test['died'], prob_tree_test), 3))

**» Make** — **The final test.** Fit your chosen model on all the training rows, and report its AUC, sensitivity and PPV on the test rows, once. In the comment: how close is it to its cross-validated AUC, and what would you tell someone who asks how well it will work at another hospital?

In [ ]:
# How close to the cross-validated AUC, and what about another hospital:

# Your code here

**» Modify (stretch)** — **KNN without the scaler.** Section 9 of Session 4 said a model that measures distance sees unscaled features very differently. Decide what you expect, then **remove `StandardScaler()` from this pipeline** and compare.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipeline_knn = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), KNeighborsClassifier(n_neighbors=10))
scores = cross_val_score(
    pipeline_knn,
    df_train[features],
    df_train['died'],
    cv=GroupKFold(n_splits=5),
    groups=df_train['subject_id'],
    scoring='roc_auc',
)
print('Mean AUC', round(scores.mean(), 3))

**» Make (stretch)** — **Let the model decide.** In Session 4's Closing, a colleague said to give the model all 24 first-day columns. Now you can measure it: **cross-validate logistic regression with every column of `df_vitals` added to `features`, and compare with the table above.**

<details><summary><b>Hint</b></summary>

`df_model.loc[df_train.index]` gives the training rows with every column of `df_model`. The first-day columns are `df_vitals.columns`, without `subject_id` and `stay_id`. A column can be in both lists, so remove the repeats with `list(dict.fromkeys(columns))`.

</details>

In [ ]:
# Your code here

**» Make (stretch)** — **A split in time.** `anchor_year_group` says roughly when each patient was treated. **Train on patients from before 2017, test on the patients from 2017 onwards, and compare the AUC and the mean predicted risk against the observed death rate.** Does this show the model drifting? What would it not show?

<details><summary><b>Hint</b></summary>

`df_features[df_features['anchor_year_group'] < '2017 - 2019']` works because the group names sort in time order. Build a pipeline as in Part 5, and fit it on these rows.

</details>

In [ ]:
# Your code here

**» Make (stretch)** — **A long tail.** `los` has a long right tail. **Fit the model of Part 1 to `np.log1p(df_train['los'])`, turn its predictions back with `np.expm1`, and compare the RMSE and the mean absolute error with Part 1's.** Which improved, and why not both?

In [ ]:
# Your code here

## Regroup

Stop where you are: we come back together for the Closing.

If you didn't finish, finish after the session: Parts 2 and 3 matter most, and the stretch tasks are optional.

Finished early? Help a neighbour.

# 9. Closing
1. In Part 3 you tuned an alert to catch 80% of deaths, flagging about 43 patients in every 100, with roughly one flag in five true. **Would you switch it on?**
2. Which AUC would you put in a paper: **0.80** with every patient, or **0.78** without those already dead at 24 hours?

By AUC and calibration, the model from section 5 is respectable. Everything since has been about what those two numbers do not tell you, and the biggest is: **respectable for what?**

## Choosing what to optimise
| | Acting is cheap | Acting is costly |
| :-- | :-- | :-- |
| **A miss is costly** | ? | ? |
| **A miss is cheap** | ? | ? |

**» Investigate** — In your pair, **find a clinical example for each box.** What should the model be tuned to get right in each?

Choose the number your use needs from the ones you have met: **sensitivity** to miss no one, **PPV** to act only when it is likely, **flags per 100** to keep the workload bearable.

## Back to the talk
In your pair for two minutes, then together:

The Michigan Medicine evaluation of the Epic sepsis model found an AUC of **0.63**. The model missed **67%** of sepsis cases, while alerting on **18%** of all hospitalised patients. ([Wong et al., 2021](https://doi.org/10.1001/jamainternmed.2021.2626))

1. **What does 0.63 mean**, in terms of two patients?
2. **A vendor** tells you their model has an AUC of 0.8 on their data. After today, what do you ask?

**» Investigate** — Back to the haemoglobin model from section 3. Its test patients came from the same Boston hospital as its training patients. **Would you trust its 0.59 g/dL in a hospital where thalassaemia trait is common?** In thalassaemia, each red cell carries less haemoglobin than usual.

## Does our model drift?
Session 4 asked whether `temperature_missing` would transfer between settings with different recording practices. MIMIC runs from 2008 to 2022, in one hospital.

In [ ]:
missing_by_period = df_features.groupby('anchor_year_group')['temperature_missing'].mean()
missing_by_period.round(3)

**» Investigate** — **What changed in how this hospital records temperature?**

A model trained on the years before 2017 and tested on 2017 onwards keeps its AUC of about 0.80 through that change. **So can the hospital stop checking it? If not, what should it watch, and how often?**

## The whole process
Everything today, in one picture: split by patient, fit one pipeline on the training rows, and use the test rows once, at the end.

![The full pipeline: split, fit on the training rows, evaluate on the test rows](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-04_Full-Pipeline.svg)

## Wrap up
*   Test on patients the model has not seen: split by patient. Whatever learns from rows learns from the training rows, and a pipeline makes that automatic
*   A number means little without a baseline: predicting the mean for regression, "always survives" for classification
*   Accuracy can look fine while the model misses most of the patients who matter. Sensitivity, PPV and flags per 100 are what a clinician acts on, and the threshold is a clinical decision
*   AUC is the chance of ranking a patient who died above one who survived. One split is one draw, so compare models by cross-validation and use the test set once
*   A good number on your data is not a promise about someone else's hospital, or about next year
*   MIMIC data stay in your notebooks: do not share outputs, and do not paste rows into AI assistants

## Before Session 6
1. **Finish the practice**, Parts 2 and 3 at least, and check your decisions against the solutions notebook once it is released.
2. **Build a model from scratch, with your trio.** The task, and the model type your trio has, are on [Canvas](https://canvas.imperial.ac.uk/courses/8306). Submit it before the session: we compare every trio's model at the start of Session 6.
3. Reading list: R-squared and residual analysis, regularisation methods, and the SVM, decision tree and KNN classifiers.

# Self-study appendices

## Example uses of supervised learning

| Regression | Classification |
| ---------- | -------------- |
| Predict surgical procedure duration (Linear Regression) - [doi.org/10.3389/fmed.2017.00085](https://doi.org/10.3389/fmed.2017.00085) | Predict risk of postoperative pulmonary complications (Logistic Regression/Boosted Trees) - [doi.org/10.1016/S2589-7500(24)00065-7](https://doi.org/10.1016/S2589-7500(24)00065-7) |
| Predict FEV1 and FVC from chest x-ray (Deep NN) - [doi.org/10.1016/S2589-7500(24)00113-4](https://doi.org/10.1016/S2589-7500(24)00113-4) | Identify (and rule out) patients with suspected myocardial infarction (Logistic Regression) - [doi.org/10.1016/S2589-7500(24)00191-2](https://doi.org/10.1016/S2589-7500(24)00191-2) |
| Predicting length of stay after appendectomy surgery (Linear Regression) - [doi.org/10.1186/s12911-022-01884-9](https://doi.org/10.1186/s12911-022-01884-9) | Post surgery mortality prediction (Boosted Trees) - [doi.org/10.3390/bioengineering8020022](https://doi.org/10.3390/bioengineering8020022) |

## Ordinary least squares: the maths
The cost function to minimise, for candidate parameters $b$:

$$S(b) = \sum_{i=1}^n (y_i - x_i^T b)^2$$

where $x_i$ holds the feature values of the $i$-th row and $y_i$ its output. Its minimum has a closed-form solution:

$$\hat{b} = (X^TX)^{-1}X^Ty$$

![OLS Error](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-04_OLS.svg)

## Logistic regression: what it minimises
The labels are 0 or 1, never a probability, so squared error is the wrong measure. Logistic regression chooses the coefficients $b$ that make the observed labels most probable, which is the same as minimising the **log loss**:

$$L(b) = -\sum_{i=1}^n \left[ y_i \log p_i + (1 - y_i) \log (1 - p_i) \right], \quad p_i = \frac{1}{1 + e^{-x_i^T b}}$$

A confident wrong prediction, $p_i$ near 0 for a patient who died, costs a lot. Unlike OLS there is no closed-form minimum, because $b$ sits inside the sigmoid. But $L(b)$ is convex, a single valley, so an iterative search from any starting guess finds the same answer. `max_iter` caps the number of steps; the default of 100 is plenty once the features are scaled.

## Linear regression: assumptions and errors
*   The features and the output are **linearly** related
*   The residual error is normally distributed
*   Features are not **multicollinear**. A complete set of one-hot columns always adds up to 1, so a full `adm_` or `race_` set is exactly collinear: the predictions are fine, but do not read the individual coefficients

Errors in the units of the output, for $n$ rows:

$$mse = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2 \qquad rmse = \sqrt{mse}$$

## R-squared and residual analysis
*   R-squared is the share of the variability in the output that the model describes: a perfect model scores 1, and predicting the mean scores 0
*   Residual analysis looks at the errors, to check the model's assumptions: a pattern in the residuals is something the model has missed
*   The reading list has more on both

## Regularisation
*   Regularisation penalises large coefficients, which makes a model less able to overfit
*   **Lasso** (L1) penalises the sum of their absolute values, and can set some to exactly zero
*   **Ridge** (L2) penalises the sum of their squares, and shrinks them all
*   **Elastic Net** combines the two
*   scikit-learn's `LogisticRegression` applies L2 by default, so today's classifier was already regularised; `LinearRegression` is not

## Other regression methods
For a relationship that is not linear:

*   Polynomial regression
*   Neural networks, next session
*   Decision trees and random forests, Session 7
*   Bayesian regression

## Other forms of cross-validation
*   **Stratified k-fold** keeps the proportion of each class the same in every fold, which helps with rare outcomes. `StratifiedGroupKFold` does this and keeps each patient in one fold
*   **Leave-one-out** makes $k$ the number of rows: each row is the test set once

Whatever the method, split fairly and independently. Prospective multi-centre validation is the gold standard, and very few models achieve it.

## Support vector machines
*   Fitting places a boundary that separates the classes by as wide a margin as possible
*   Non-linear boundaries are possible through kernels, which map the data into higher dimensions
*   `sklearn.svm.SVC`

![Illustration of a SVM](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-04_SVM.svg)

## Decision trees
*   Data are split into branches on feature values, leading to a classification at each leaf
*   Can model non-linear relationships, and the fitted model can be read by a person
*   Without a limit on depth, a tree will memorise its training rows
*   `sklearn.tree.DecisionTreeClassifier`

![Illustration of a decision tree](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-04_Decision-Tree.svg)

## K-nearest neighbours
*   Predicts a patient's class from the $k$ closest rows in the training data
*   Can model non-linear relationships, and is simple to explain
*   It works on distances, so scaling matters
*   `sklearn.neighbors.KNeighborsClassifier`

![Illustration of KNN](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-04_KNN.svg)